In [17]:
import pandas as pd
import ast
import numpy as np
import os
import pronto
from itertools import islice
from tqdm import tqdm
import json
from transformers import AutoTokenizer, AutoModel


## Entity normalization tests

### Load MONDO terminology
Note: 
- https://mondo.monarchinitiative.org/pages/download/

In [2]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)


In [3]:
%%capture

# Load the ontology
ontology = pronto.Ontology("data/mondo/mondo.owl")


In [24]:
# 'neurotic depression', 'MONDO:0024614', ['relapsing-remitting multiple sclerosis', 'MONDO:0005314']
term = ontology["MONDO:0005314"]  # obsolete 46,XX sex reversal

# Get all superclasses (i.e., recursive parent terms)
print("📚 All ancestor terms:")
for parent in term.superclasses():
    print(parent.id, "-", parent.name)

📚 All ancestor terms:
MONDO:0005314 - relapsing-remitting multiple sclerosis
MONDO:0005301 - multiple sclerosis
MONDO:0000568 - autoimmune disorder of central nervous system
MONDO:0005560 - brain disorder
MONDO:0006704 - CNS demyelinating autoimmune disease
MONDO:0020800 - demyelinating disease of central nervous system
MONDO:0002602 - central nervous system disorder
MONDO:0002977 - autoimmune disorder of the nervous system
MONDO:0007179 - autoimmune disease
MONDO:0002562 - demyelinating disease
MONDO:0005071 - nervous system disorder
MONDO:0005046 - immune system disorder
MONDO:0005559 - neurodegenerative disease
MONDO:0021147 - disorder of development or morphogenesis
MONDO:0700096 - human disease
MONDO:0000001 - disease
BFO:0000016 - disposition
BFO:0000017 - realizable entity
BFO:0000020 - specifically dependent continuant
BFO:0000002 - continuant
BFO:0000001 - entity


In [27]:
print("Immediate parents:")
for parent in term.superclasses(distance=1):
    print(parent.id, "-", parent.name)

Immediate parents:
MONDO:0005314 - relapsing-remitting multiple sclerosis
MONDO:0005301 - multiple sclerosis


### Load embeddings

In [7]:
# Define the directory where your files are stored
load_snomed = True
if load_snomed:
    directory_path = "./data/snomed/embeddings"
    batch_name_prefix = "disorder_substance_finding_emb"
else:
    directory_path = "./data/mondo/embeddings"
    batch_name_prefix = "MONDO_emb"

# List all files in the directory that match the pattern
files = [f for f in os.listdir(directory_path) if f.startswith(f'{batch_name_prefix}_batch_') and f.endswith('.npy')]
print(files)
# Sort files to maintain the order, especially important if the batch index is used in processing
files.sort()

# Initialize an empty list to hold the data from each file
all_data = []

# Load each file and append the data to the list
for file in files:
    file_path = os.path.join(directory_path, file)
    data = np.load(file_path)
    all_data.append(data)

if load_snomed:
    # Concatenate all the arrays from the list into one
    all_reps_emb_full_snomed = np.concatenate(all_data, axis=0)
else:
    all_reps_emb_full = np.concatenate(all_data, axis=0)

['disorder_substance_finding_emb_batch_0.npy', 'disorder_substance_finding_emb_batch_1.npy', 'disorder_substance_finding_emb_batch_3.npy', 'disorder_substance_finding_emb_batch_2.npy', 'disorder_substance_finding_emb_batch_4.npy']


In [8]:
all_reps_emb_full.shape

(30084, 768)

In [9]:
all_reps_emb_full_snomed.shape

(408651, 768)

In [10]:
with open("data/snomed/disorder_substance_finding_snomed_sf_id_pairs.json", "r") as f:
    snomed_sf_id_pairs = json.load(f)

In [11]:
with open("data/mondo/mondo_term_id_pairs.json", "r") as f:
    term_id_pairs = json.load(f)

# Test mappings

In [18]:
tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
model = AutoModel.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")

In [12]:
import numpy as np
import torch
from transformers import PreTrainedTokenizer, PreTrainedModel
from typing import Any, Tuple, Dict, List, Union
from scipy.spatial.distance import cdist


def map_query_to_terminology(query: str, 
                        tokenizer: PreTrainedTokenizer, 
                        model: PreTrainedModel, 
                        all_reps_emb_full: np.ndarray, 
                        snomed_sf_id_pairs: np.ndarray, 
                        canonical_mapping_dict: Dict[str, str] = None,
                        dist_threshold: float = 15,  # Added threshold parameter
                        n_entities: int = 5) -> Tuple[int, str, str, List[Tuple[str, int]], float]:
    
    """
    Map a query to the closest SNOMED concept using a pre-trained model and return its canonical form.
    If the distance to the nearest concept exceeds the threshold, return the original query.

    Parameters:
    - query (str): The input query string to be mapped.
    - tokenizer (PreTrainedTokenizer): The tokenizer used for encoding the query.
    - model (PreTrainedModel): The pre-trained model used to generate embeddings for the query.
    - all_reps_emb_full (np.ndarray): The array of embeddings for all SNOMED concepts.
    - snomed_sf_id_pairs (np.ndarray): The array of SNOMED concept ID and label pairs.
    - canonical_mapping_dict (Dict[str, str]): A dictionary mapping SNOMED IDs to their canonical forms.
    - dist_threshold (float): Distance threshold for deciding whether to map the query.
    - n_entities (int): The number of nearest entities to retrieve.

    Returns:
    - Tuple[int, str, str, List[Tuple[str, int]], float]: The predicted SNOMED concept ID or 0 for no mapping,
      label, its canonical form or original query, a list of the nearest entities, and the minimum distance.
    """
    # Move embeddings to GPU if available
    if torch.cuda.is_available():
        all_reps_emb_full= torch.tensor(all_reps_emb_full).to('cuda')

    if torch.cuda.is_available():
        model = model.to('cuda')
        
    # Encode the query
    query_toks = tokenizer.batch_encode_plus([query], 
                                             padding="max_length", 
                                             max_length=25, 
                                             truncation=True,
                                             return_tensors="pt")
    if torch.cuda.is_available():
        query_toks = query_toks.to('cuda')  # Move tensors to GPU
        
    # Get the model output
    with torch.no_grad():
        query_output = model(**query_toks)
    
    # Extract the CLS token representation
    query_cls_rep = query_output[0][:, 0, :]

    # Compute distances between query embedding and all SNOMED concept embeddings
    if torch.cuda.is_available():
        dist = torch.cdist(query_cls_rep, all_reps_emb_full)
        nn_index = torch.argmin(dist).item()  # This finds the index of the minimum value
        min_distance = dist[0, nn_index].item()  # Extract the minimum distance at that index
    else:
        dist = cdist(query_cls_rep.cpu().detach().numpy(), all_reps_emb_full)
        nn_index = np.argmin(dist).item()
        min_distance = dist[0, nn_index]  # Since dist is a numpy array, get the minimum distance at the index 

    # Retrieve the nearest n_entities
    nearest_n_entities = []
    if torch.cuda.is_available():
        nearest_n_indices = torch.argsort(dist[0])[:n_entities]  # Get indices of the n smallest distances
    else:
        nearest_n_indices = np.argsort(dist[0])[:n_entities]
    for idx in nearest_n_indices:
        nearest_n_entities.append(snomed_sf_id_pairs[idx.item()])

    if min_distance > dist_threshold:
        # If distance is greater than the threshold, return the original query with no SNOMED mapping
        return -1, query, query, nearest_n_entities, round(min_distance, 4)
        
    # Get the predicted SNOMED concept ID and label
    predicted_label = snomed_sf_id_pairs[nn_index]
    predicted_id = predicted_label[1]

    # Get the canonical form from the dictionary
    if canonical_mapping_dict:
        canonical_form = canonical_mapping_dict.get(predicted_id, "Canonical form not found")
    else:
        canonical_form = predicted_label

    # Return the predicted SNOMED concept ID, label, and canonical form
    return predicted_id, predicted_label[0], canonical_form, nearest_n_entities, round(min_distance, 4)


/opt/anaconda3/envs/preclinical/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
len(all_reps_emb_full), len(term_id_pairs)

(30084, 30084)

In [28]:
# Example usage
dist_threshold = 15
map_to_snomed = False
if map_to_snomed:
    embeddings_to_use = all_reps_emb_full_snomed
    corresponding_term_id = snomed_sf_id_pairs
else:
    embeddings_to_use = all_reps_emb_full
    corresponding_term_id = term_id_pairs
    
query = "relapsing remitting ms lkdjf"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: -1
Predicted label: relapsing remitting ms lkdjf
Distance: 11.8354
Canonical form: relapsing remitting ms lkdjf
Nearest 3:  [['relapsing-remitting multiple sclerosis', 'MONDO:0005314'], ['progressive relapsing multiple sclerosis', 'MONDO:0000452'], ['Marburg acute multiple sclerosis', 'MONDO:0016429'], ['leukodystrophy, childhood-onset, remitting', 'MONDO:0859246'], ['metabolic crises, recurrent, with variable encephalomyopathic features and neurologic regression', 'MONDO:0032736']]


In [74]:
id_to_term['MONDO:0011513']

Term('MONDO:0011513', name='Alzheimer disease, familial early-onset, with coexisting amyloid and prion pathology')